# 21.1 PySpark 入门:分布式计算的思维模型 / PySpark Basics: the mental model of distributed computing

**中文**:当数据从 GB 涨到 **TB / PB**,单机的 pandas 会因为**内存装不下**而直接崩溃。这时必须把数据和计算**切分到一整个集群的很多台机器上并行处理**——这就是 **Apache Spark** 的战场:大数据处理的事实标准,几乎每个数据/ML 岗位都会问。但很遗憾,很多教程只教你"调 API",却没讲清楚 Spark 背后**为什么这样设计**。本节反其道:我们**从零用纯 Python 实现一个 MiniSpark**(几十行),把 Spark 最核心的三个思想——**分区并行(partitioned parallelism)、惰性求值(lazy evaluation)、宽窄依赖与 Shuffle**——变成你能亲手运行、亲眼看到的东西。理解了这个思维模型,真实的 PySpark API 就只是语法细节。
**English**: When data grows from GB to **TB / PB**, single-machine pandas simply crashes because **it can't fit in memory**. You must then **split data and computation across many machines in a cluster and process in parallel** — this is **Apache Spark**'s arena: the de-facto standard for big-data processing, asked in nearly every data/ML interview. Unfortunately, many tutorials only teach "which API to call" without explaining **why Spark is designed the way it is**. We do the opposite: we **build a MiniSpark from scratch in pure Python** (a few dozen lines) that turns Spark's three core ideas — **partitioned parallelism, lazy evaluation, and narrow/wide dependencies & Shuffle** — into things you can run and see with your own eyes. Once you grasp this mental model, the real PySpark API is just syntax.

---

**中文**:**Spark 的架构(一定要会画)**:
**English**: **Spark's architecture (you must be able to draw it)**:
- **中文**:**Driver(驱动器)**:你的主程序,构建计算计划(DAG)、把任务派发给执行器、收集结果。**一个应用一个 Driver**。
  **Driver**: your main program; builds the computation plan (DAG), dispatches tasks to executors, collects results. **One Driver per application.**
- **中文**:**Executors(执行器)**:分布在集群各机器上的工作进程,**真正干活**——每个执行器处理数据的若干**分区(partition)**,并把结果缓存在自己的内存里。
  **Executors**: worker processes spread across the cluster machines that **do the actual work** — each processes several **partitions** of the data and caches results in its own memory.
- **中文**:**Cluster Manager(集群管理器)**:YARN / Kubernetes / Standalone,负责给应用分配机器资源。
  **Cluster Manager**: YARN / Kubernetes / Standalone; allocates machine resources to applications.
- **中文**:**Partition(分区)是并行的基本单位**——一个 1TB 文件被切成比如 8000 个分区,分散到执行器上,每个分区独立、并行地被处理。**分区数≈并行度**。
  **The Partition is the unit of parallelism** — a 1TB file is split into, say, 8000 partitions spread across executors, each processed independently and in parallel. **#partitions ≈ degree of parallelism.**

**中文**:**惰性求值(lazy evaluation)是 Spark 的灵魂**。Spark 把操作分两类:
**English**: **Lazy evaluation is Spark's soul.** Spark splits operations into two kinds:
- **中文**:**转换(transformations)**:`map`/`filter`/`join`/`groupBy`……**只构建计算计划(DAG),不真正执行**。它们是"惰性"的。
  **Transformations**: `map`/`filter`/`join`/`groupBy`… **only build the computation plan (DAG); they don't actually run.** They are "lazy."
- **中文**:**动作(actions)**:`collect`/`count`/`save`/`show`……**才真正触发整个 DAG 的计算**。
  **Actions**: `collect`/`count`/`save`/`show`… **actually trigger execution of the whole DAG.**

**中文**:为什么惰性?因为看到完整的计划后,Spark 能**全局优化**——合并连续的 `map`、把 `filter` 下推到读数据时、避免不必要的中间结果落地。这就是 **Catalyst 优化器**的基础。
**English**: Why lazy? Because seeing the full plan lets Spark **globally optimize** — fuse consecutive `map`s, push `filter`s down to the data read, avoid materializing unnecessary intermediates. This is the basis of the **Catalyst optimizer**.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 大数据必考, 几乎逢面必问）**
> **中文**:**Spark 架构**:Driver(建 DAG+调度)+ Executors(处理分区、真正计算)+ Cluster Manager(YARN/K8s 分配资源); **分区=并行单位**。**惰性求值**:transformations(map/filter/join, 只建 DAG)vs actions(collect/count/save, 才触发计算)→ 让 Catalyst 全局优化。**RDD vs DataFrame vs Dataset**:RDD(底层, 无 schema, 无优化, 灵活)；**DataFrame(有 schema, 走 Catalyst 优化+Tungsten 内存管理, 首选)**；Dataset(Scala/Java 强类型)。**窄依赖 vs 宽依赖**:窄(map/filter, 一父分区→一子分区, 无需 shuffle, 可流水线)；**宽(groupByKey/join/reduceByKey, 跨分区重排=Shuffle, 昂贵, 划分 Stage 边界)**。**Shuffle 是最大性能杀手**(磁盘+网络+序列化)→ 减少 shuffle、用 reduceByKey 而非 groupByKey、broadcast join 小表。**惰性坑**:transformations 不执行 → 忘了 action 什么都不会发生; 多次 action 会重算 → 用 `cache()/persist()`。**何时别用 Spark**:数据能塞进单机内存时(<几十GB), pandas/Polars/DuckDB 更快更简单——Spark 的分布式开销只有大数据才划算。
> **English**: **Spark architecture**: Driver (builds DAG + schedules) + Executors (process partitions, do the real compute) + Cluster Manager (YARN/K8s allocate resources); **partition = unit of parallelism**. **Lazy evaluation**: transformations (map/filter/join, only build the DAG) vs actions (collect/count/save, actually trigger compute) → lets Catalyst globally optimize. **RDD vs DataFrame vs Dataset**: RDD (low-level, no schema, no optimization, flexible); **DataFrame (has schema, goes through Catalyst optimization + Tungsten memory management, the default choice)**; Dataset (Scala/Java strongly-typed). **Narrow vs wide dependency**: narrow (map/filter, one parent partition → one child partition, no shuffle, pipelineable); **wide (groupByKey/join/reduceByKey, cross-partition reshuffle = Shuffle, expensive, marks Stage boundaries)**. **Shuffle is the biggest performance killer** (disk + network + serialization) → minimize shuffles, use reduceByKey over groupByKey, broadcast-join small tables. **Lazy gotchas**: transformations don't execute → forget an action and nothing happens; multiple actions recompute → use `cache()/persist()`. **When NOT to use Spark**: when data fits in single-machine memory (<tens of GB), pandas/Polars/DuckDB are faster and simpler — Spark's distributed overhead only pays off on big data.


In [ ]:

# ============================================================
# 从零实现 MiniSpark:惰性、分区、宽窄依赖 / MiniSpark from scratch
# 中文:一个 RDD 持有若干"分区"(每个分区=会在一个执行器上处理的一段数据)。转换是惰性的
#      (只记录计算逻辑到 DAG), 动作才真正触发计算。这正是真实 Spark 的执行模型。
# English: an RDD holds several "partitions" (each = a slice processed on one executor). Transformations are lazy
#      (they just record the computation into a DAG); actions actually trigger it. This IS real Spark's model.
# ============================================================
from functools import reduce as _reduce
class MiniRDD:
    def __init__(self, partitions, lineage="parallelize", parents=None, wide=False):
        self._partitions=partitions          # 分区数据, 或一个"惰性"返回分区的函数 / partition data OR a lazy fn
        self.lineage=lineage                 # 这一步是什么操作 / what op this step is
        self.parents=parents or []           # 上游 RDD(构成 DAG)/ upstream RDDs (form the DAG)
        self.wide=wide                        # 是否宽依赖(需要 shuffle)/ wide dependency (needs shuffle)?
    def _parts(self):                         # 求值:如果是惰性函数就现在执行 / materialize partitions now
        return self._partitions() if callable(self._partitions) else self._partitions
    # ---------- 转换 transformations:惰性, 只把逻辑挂到 DAG 上 / lazy, just attach to the DAG ----------
    def map(self,f):     return MiniRDD(lambda:[[f(x)  for x in p]          for p in self._parts()],"map",[self])      # 窄依赖
    def filter(self,f):  return MiniRDD(lambda:[[x     for x in p if f(x)]  for p in self._parts()],"filter",[self])   # 窄依赖
    def flatMap(self,f): return MiniRDD(lambda:[[y for x in p for y in f(x)]for p in self._parts()],"flatMap",[self])  # 窄依赖
    def reduceByKey(self,f,n=None):
        def shuffle():                        # 宽依赖:按 key 的哈希把数据重排到新分区(这就是 Shuffle!)/ WIDE: reshuffle by key hash
            parts=self._parts(); npart=n or len(parts); buckets=[dict() for _ in range(npart)]
            for p in parts:
                for k,v in p:
                    b=hash(k)%npart           # key 决定去哪个新分区 / key hash decides target partition
                    buckets[b][k]=f(buckets[b][k],v) if k in buckets[b] else v
            return [list(b.items()) for b in buckets]
        return MiniRDD(shuffle,"reduceByKey",[self],wide=True)   # wide=True → 这里是一个 Stage 边界 / a Stage boundary
    # ---------- 动作 actions:真正触发整个 DAG 计算 / actually run the whole DAG ----------
    def collect(self): return [x for p in self._parts() for x in p]
    def count(self):   return sum(len(p) for p in self._parts())
    def reduce(self,f):return _reduce(f,[x for p in self._parts() for x in p])
    def toDAG(self,d=0):                       # 打印 DAG / print the DAG (lineage)
        line="  "*d+("└─" if d else "")+self.lineage+(" [SHUFFLE—宽依赖, Stage 边界]" if self.wide else "")
        return line+"".join("\n"+p.toDAG(d+1) for p in self.parents)

def parallelize(data,n=4):                     # 把数据切成 n 个分区 / split data into n partitions
    return MiniRDD([data[i::n] for i in range(n)])
print("MiniRDD 定义完成 / MiniRDD defined. 分区=并行单位, 转换=惰性, 动作=触发计算")


In [ ]:

# ============================================================
# 演示 1:惰性求值——构建转换时什么都不会算 / Demo 1: lazy evaluation
# ============================================================
sc = parallelize(list(range(1,13)), n=4)                 # 12 个数字切成 4 个分区 / 12 numbers into 4 partitions
print("原始 4 个分区 / 4 partitions:", sc._parts())
pipeline = sc.filter(lambda x: x%2==0).map(lambda x: x*x) # 只构建 DAG, 一行都没真正执行! / builds DAG only, runs nothing
print("\n构建了转换但还没触发计算。DAG(计算血缘)如下 / built transformations, nothing ran yet. DAG:")
print(pipeline.toDAG())
print("\n现在调用动作 collect() —— 才真正从头执行整个 DAG / calling action collect() NOW triggers the whole DAG:")
print("  collect() =", pipeline.collect())               # 偶数的平方 / squares of evens
print("  count()   =", pipeline.count())


In [ ]:

# ============================================================
# 演示 2:WordCount(大数据界的 "Hello World")+ 观察 Shuffle / Demo 2: WordCount + observe the Shuffle
# 中文:词频统计需要把"同一个词"聚到一起→跨分区重排=Shuffle=宽依赖=Stage 边界。这是分布式最贵的操作。
# English: counting words must gather "the same word" together → cross-partition reshuffle = Shuffle = wide dep = Stage boundary.
# ============================================================
lines = ["the cat sat on the mat","the dog ran fast","a cat and a dog","the the the"]
wc = (parallelize(lines, n=3)                            # 文本切成 3 个分区(3 个执行器)/ text into 3 partitions
        .flatMap(lambda line: line.split())              # 每行拆成词(窄)/ split lines into words (narrow)
        .map(lambda w: (w, 1))                           # 每个词→(词,1)(窄)/ each word → (word,1) (narrow)
        .reduceByKey(lambda a,b: a+b))                   # 按词求和(宽依赖=Shuffle!)/ sum by word (WIDE = Shuffle!)
print("WordCount 的 DAG —— 注意 reduceByKey 处的 Shuffle 把计算切成两个 Stage:")
print("WordCount DAG — note the Shuffle at reduceByKey splits computation into two Stages:")
print(wc.toDAG())
print("\n结果 / result:", dict(sorted(wc.collect(), key=lambda kv:-kv[1])))


In [ ]:

# ============================================================
# 可视化:Spark 架构 + 宽窄依赖 / Visualization: architecture + narrow/wide deps
# ============================================================
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(15,5.5))
# ① 架构图 / architecture
ax[0].axis("off"); ax[0].set_title("Spark 架构:Driver → Executors → Partitions",fontsize=12,weight="bold")
ax[0].add_patch(plt.Rectangle((0.35,0.82),0.3,0.13,fc="#4C72B0",alpha=0.7,transform=ax[0].transAxes))
ax[0].text(0.5,0.885,"Driver\n(建 DAG + 调度)",ha="center",va="center",color="white",fontsize=10,transform=ax[0].transAxes)
for i,x0 in enumerate([0.05,0.38,0.71]):
    ax[0].add_patch(plt.Rectangle((x0,0.45),0.24,0.18,fc="#55A868",alpha=0.7,transform=ax[0].transAxes))
    ax[0].text(x0+0.12,0.54,f"Executor {i+1}",ha="center",va="center",color="white",fontsize=9,transform=ax[0].transAxes)
    ax[0].annotate("",xy=(x0+0.12,0.63),xytext=(0.5,0.82),arrowprops=dict(arrowstyle="->",color="gray"),transform=ax[0].transAxes)
    for j in range(2):
        ax[0].add_patch(plt.Rectangle((x0+0.02+j*0.11,0.28),0.09,0.12,fc="#DD8452",alpha=0.6,transform=ax[0].transAxes))
        ax[0].text(x0+0.065+j*0.11,0.34,f"P{i*2+j}",ha="center",va="center",fontsize=8,transform=ax[0].transAxes)
ax[0].text(0.5,0.12,"分区(P)=并行单位, 每个在一个执行器上独立处理\npartition = unit of parallelism, each processed on an executor",
           ha="center",fontsize=9,style="italic",transform=ax[0].transAxes)
# ② 宽窄依赖 / narrow vs wide
ax[1].axis("off"); ax[1].set_title("窄依赖(可流水线)vs 宽依赖(Shuffle)",fontsize=12,weight="bold")
# narrow
for i in range(3):
    ax[1].add_patch(plt.Rectangle((0.05,0.7-i*0.12),0.12,0.09,fc="#55A868",alpha=0.6,transform=ax[1].transAxes))
    ax[1].add_patch(plt.Rectangle((0.28,0.7-i*0.12),0.12,0.09,fc="#55A868",alpha=0.6,transform=ax[1].transAxes))
    ax[1].annotate("",xy=(0.28,0.745-i*0.12),xytext=(0.17,0.745-i*0.12),arrowprops=dict(arrowstyle="->",color="green"),transform=ax[1].transAxes)
ax[1].text(0.23,0.86,"窄 narrow (map/filter)",ha="center",fontsize=9,color="green",transform=ax[1].transAxes)
ax[1].text(0.23,0.28,"一父→一子, 无需网络\n1 parent→1 child, no network",ha="center",fontsize=8,transform=ax[1].transAxes)
# wide (shuffle)
for i in range(3):
    ax[1].add_patch(plt.Rectangle((0.55,0.7-i*0.12),0.12,0.09,fc="#C44E52",alpha=0.6,transform=ax[1].transAxes))
    for k in range(3):
        ax[1].add_patch(plt.Rectangle((0.8,0.7-k*0.12),0.12,0.09,fc="#C44E52",alpha=0.6,transform=ax[1].transAxes))
        ax[1].annotate("",xy=(0.8,0.745-k*0.12),xytext=(0.67,0.745-i*0.12),arrowprops=dict(arrowstyle="->",color="#C44E52",alpha=0.4),transform=ax[1].transAxes)
ax[1].text(0.73,0.86,"宽 wide (groupBy/join)",ha="center",fontsize=9,color="#C44E52",transform=ax[1].transAxes)
ax[1].text(0.73,0.28,"多父→多子=Shuffle\n跨网络重排, 最贵!\ncross-network reshuffle, expensive!",ha="center",fontsize=8,transform=ax[1].transAxes)
plt.tight_layout(); plt.savefig("/tmp/big01_viz.png",dpi=80); plt.show()
print("左:分区分散到执行器并行; 右:窄依赖可本地流水线, 宽依赖必须 Shuffle(跨网络, Stage 边界)")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **Spark 的强大来自三个简单思想的组合,而非魔法**:①**分区并行**——把大数据切成很多分区,每个分区在一个执行器上独立处理,机器越多越快(水平扩展);②**惰性求值**——转换只建计划、动作才执行,让引擎看到全局后统一优化;③**血缘(lineage)** —— DAG 记录了"每个分区是怎么从原始数据算出来的",所以**某台机器挂了,只需按血缘重算那几个分区,而非整个作业**(容错)。我们的 MiniSpark 用几十行就复现了这三点——真实 Spark 只是在此之上加了分布式调度、内存管理、SQL 优化。理解了模型,API 就是细节。
2. **Shuffle 是分布式计算的头号性能敌人**:窄依赖(map/filter)可以在本地流水线执行,一个分区的数据不用离开所在机器;但宽依赖(groupByKey/join/reduceByKey)需要**把数据按 key 跨机器重新分发**——这涉及磁盘写、网络传、序列化,是**最慢、最容易 OOM、最容易数据倾斜**的环节。我们的 `reduceByKey` 里那个 `hash(k)%npart` 的重排,正是 Shuffle 的本质。90% 的 Spark 调优都在**减少 Shuffle**:用 `reduceByKey` 替代 `groupByKey`(前者先本地聚合再传)、广播小表避免 join shuffle、合理设分区数。
3. **诚实的边界:大多数人根本不需要 Spark**。Spark 的分布式能力有**沉重的固定开销**——启动集群、序列化、网络往返、JVM 调优。当数据能塞进单机内存(现在一台机器几百 GB 内存很常见),**Polars / DuckDB / pandas 往往比 Spark 快几倍且简单得多**(后面 21.6/21.7 会实测)。行业里一个著名的教训是:**很多公司为了"显得高大上"上了 Spark 集群处理其实只有几 GB 的数据,结果又慢又贵又难维护**。正确的判断:**先问数据到底多大**——单机内存放得下就别用 Spark;真到 TB/PB、需要横跨几百台机器时,Spark(或云上的托管 Spark)才是对的工具。**用对规模的工具,是数据工程最重要的判断力之一。**

**English**:
1. **Spark's power comes from combining three simple ideas, not magic**: ① **partitioned parallelism** — split big data into many partitions, each processed independently on an executor, faster with more machines (horizontal scaling); ② **lazy evaluation** — transformations only build a plan, actions execute, letting the engine optimize globally; ③ **lineage** — the DAG records "how each partition was computed from the raw data," so **if a machine dies, only those few partitions are recomputed via lineage, not the whole job** (fault tolerance). Our MiniSpark reproduces all three in a few dozen lines — real Spark just adds distributed scheduling, memory management, and SQL optimization on top. Understand the model and the API is detail.
2. **Shuffle is distributed computing's #1 performance enemy**: narrow dependencies (map/filter) pipeline locally, a partition's data never leaving its machine; but wide dependencies (groupByKey/join/reduceByKey) must **redistribute data by key across machines** — involving disk writes, network transfer, and serialization, the **slowest, most OOM-prone, most data-skew-prone** stage. That `hash(k)%npart` reshuffle in our `reduceByKey` is exactly the essence of Shuffle. 90% of Spark tuning is **reducing Shuffle**: use `reduceByKey` over `groupByKey` (the former pre-aggregates locally before transferring), broadcast small tables to avoid join shuffles, set partition counts sensibly.
3. **Honest limits: most people don't need Spark at all.** Spark's distributed power carries **heavy fixed overhead** — cluster startup, serialization, network round-trips, JVM tuning. When data fits in single-machine memory (hundreds of GB of RAM per machine is now common), **Polars / DuckDB / pandas are often several times faster and far simpler than Spark** (benchmarked later in 21.6/21.7). A famous industry lesson: **many companies spun up Spark clusters to "look sophisticated" for data that was really only a few GB — ending up slower, costlier, and harder to maintain**. The right judgment: **first ask how big the data really is** — if it fits in single-machine memory, don't use Spark; only at TB/PB spanning hundreds of machines is Spark (or managed cloud Spark) the right tool. **Using the right tool for the scale is one of the most important judgments in data engineering.**

> 💼 **实战视角 / Practical angle**
> **中文**:PySpark 落地:①**入口** `SparkSession.builder.getOrCreate()`, 读数据 `spark.read.parquet(...)`, **优先用 DataFrame API 而非 RDD**(有 Catalyst 优化);②**调优看 Spark UI**——找 Shuffle 大、数据倾斜(某个 task 特别慢)、spill(内存不够写磁盘);③**减 Shuffle**:`reduceByKey`>`groupByKey`、`broadcast(small_df)` join 小表、`repartition/coalesce` 控分区、避免 `collect()` 把大数据拉回 Driver(会 OOM);④**缓存复用** `df.cache()`(多次 action 时);⑤**分区裁剪 + 列裁剪**(用 Parquet, 只读需要的列/分区)。云上多用托管 Spark(Databricks、EMR、Dataproc)。面试金句:*"Spark 靠分区并行+惰性求值+血缘容错处理大数据; 转换建 DAG、动作触发计算; 宽依赖(join/groupBy)需要 Shuffle 是最大性能瓶颈, 优化核心是减 Shuffle(reduceByKey、broadcast join)、避免数据倾斜和 collect 到 Driver; 但数据能进单机内存时 Polars/DuckDB 更快, 别为小数据上 Spark。"*
> **English**: PySpark in practice: ① **entry** `SparkSession.builder.getOrCreate()`, read with `spark.read.parquet(...)`, **prefer the DataFrame API over RDD** (Catalyst optimization); ② **tune via the Spark UI** — find big Shuffles, data skew (one task far slower), spills (out-of-memory writes to disk); ③ **reduce Shuffle**: `reduceByKey` > `groupByKey`, `broadcast(small_df)` to join small tables, `repartition/coalesce` to control partitions, avoid `collect()` pulling big data back to the Driver (OOM); ④ **cache for reuse** `df.cache()` (with multiple actions); ⑤ **partition pruning + column pruning** (use Parquet, read only needed columns/partitions). In the cloud use managed Spark (Databricks, EMR, Dataproc). Interview line: *"Spark handles big data via partitioned parallelism + lazy evaluation + lineage fault tolerance; transformations build the DAG, actions trigger compute; wide dependencies (join/groupBy) need a Shuffle, the biggest bottleneck, so optimization centers on reducing Shuffle (reduceByKey, broadcast join), avoiding data skew and collect-to-Driver; but when data fits single-machine memory, Polars/DuckDB are faster — don't use Spark for small data."*

---
### 小结 / Summary
- **中文**:Spark=分区并行 + 惰性求值 + 血缘容错; Driver 建 DAG 调度, Executors 处理分区真正计算。
- **English**: Spark = partitioned parallelism + lazy evaluation + lineage fault tolerance; Driver builds DAG & schedules, Executors process partitions and do the real compute.
- **中文**:转换(惰性建 DAG)vs 动作(触发计算); 窄依赖可流水线, 宽依赖=Shuffle(最贵, Stage 边界)。
- **English**: Transformations (lazy, build DAG) vs actions (trigger compute); narrow deps pipeline, wide deps = Shuffle (most expensive, Stage boundary).
- **中文**:DataFrame(Catalyst 优化)优于 RDD; 数据能进单机内存就别用 Spark(Polars/DuckDB 更快)。
- **English**: DataFrame (Catalyst-optimized) beats RDD; if data fits single-machine memory, skip Spark (Polars/DuckDB faster).
